# Phase 2 plumbing test — real InternVL2-2B on placeholder fixtures

**Purpose.** Prove the new `InternVL2Adapter` actually loads on a real GPU, preprocesses an image, runs `model.chat()`, and emits parseable output through the existing Phase 2 single-pass runner.

**What this run is NOT.** This is *not* a research result. The fixture pages are 8×8 placeholder PNGs; the model output will not be meaningful. The point is structural: every step of the pipeline executes against the real model. Real OmniDocBench data comes in a later step.

**Required runtime.** Colab Pro — any GPU (L4/A100/T4/V100). The adapter auto-picks dtype (`bfloat16` on Ampere+, `float16` elsewhere). InternVL2-2B is ~4 GB and fits comfortably in 16 GB VRAM.

Run cells top to bottom. If something fails, the troubleshooting section at the bottom maps common errors to fixes.

## 1. GPU sanity check

If this prints `CPU only`, switch the runtime: `Runtime → Change runtime type → GPU` and re-run.

In [ ]:
import subprocess
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
    print(out)
except Exception:
    print("CPU only — switch to GPU runtime before continuing.")

## 2. Clone the repo

If your fork URL differs, edit `GIT_URL` below. Default is the public main branch.

In [ ]:
import os, subprocess
GIT_URL = "https://github.com/Michaelhamaty/Resarch_dev.git"
BRANCH  = "main"
REPO_DIR = "/content/Research_claude"

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists — pulling latest")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, GIT_URL, REPO_DIR])

os.chdir(REPO_DIR)
print("\nWorking dir:", os.getcwd())
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 3. Install dependencies

Two layers:
1. The repo itself (editable install — picks up `pyproject.toml`).
2. The deep-learning stack required by the InternVL2 adapter.

We pin `transformers==4.40.0` because newer versions sometimes break trust-remote-code checkpoints, and older ones lack APIs InternVL2 expects.

We deliberately do NOT install `flash-attn` (slow to compile on Colab; InternVL2-2B works without it).

In [ ]:
!pip install -q -e .
!pip install -q "transformers==4.40.0" "timm>=0.9" "einops>=0.7" "sentencepiece>=0.1.99" "accelerate>=0.27" "protobuf>=3.20"

## 4. Verify tests still pass on this host

All 251 tests must pass before we touch the real model. If any fail, do not proceed — the env is wrong.

In [ ]:
!python -m pytest -q

## 5. Build Phase 1 manifests + placeholder fixture images

Both scripts are idempotent. They (re)create `data/splits/*.json` and `data/fixtures/images/page_*.png`.

In [ ]:
!python scripts/subset_extraction/build_phase1_manifests.py --config configs/dataset/phase1.yaml
!python scripts/fixtures/generate_placeholder_images.py

## 6. Smoke-load the model alone (catches HF / transformers errors before the runner runs)

Loading the model takes ~1–2 minutes the first time (downloading weights ~4 GB) and a few seconds on subsequent runs (cached at `~/.cache/huggingface`). If this cell fails, the error is in the model loader, not the project pipeline.

In [ ]:
from adaptive_inference.config.models import load_model_config
from adaptive_inference.inference.factory import build_adapter

cfg = load_model_config("configs/models/internvl2_real.yaml", "internvl2-2b")
print("Loading", cfg.model_id, "— first run downloads ~4 GB")
adapter = build_adapter(cfg)
print("✅ Loaded. device:", adapter.device, "dtype:", adapter.dtype)

## 7. Single-page smoke call (catches generation errors before the multi-page run)

Run the adapter on one fixture page directly. This isolates problems with `model.chat()` from the runner harness.

In [ ]:
from PIL import Image
from adaptive_inference.config.budgets import load_budget
from adaptive_inference.config.prompts import load_prompt_template

budget = load_budget("configs/budgets/phase2.yaml", "low")
prompt = load_prompt_template("configs/prompts/table_parse_v1.yaml")
img = Image.open("data/fixtures/images/page_0001.png").convert("RGB")

result = adapter.run(page_id="page_0001", image=img, budget=budget, prompt=prompt)
print("✅ model.chat() returned in", round(result.runtime_ms, 1), "ms")
print("output_token_count:", result.output_token_count)
print("---raw_text (first 500 chars)---")
print(result.raw_text[:500])

## 8. Run the full Phase 2 pipeline on the calibration split (5 pages)

This drives the adapter through the existing `run_single_pass` orchestrator using `configs/runs/colab_real_2b_low.yaml`. Outputs land at `outputs/runs/colab_real_2b_low_v1/`.

In [ ]:
!python scripts/main_runs/run_single_pass.py --config configs/runs/colab_real_2b_low.yaml

## 9. Inspect the artifact tree

In [ ]:
import os
RUN_DIR = "outputs/runs/colab_real_2b_low_v1"
for root, dirs, files in os.walk(RUN_DIR):
    indent = "  " * (root.count(os.sep) - RUN_DIR.count(os.sep))
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}  {f}  ({size} B)")

In [ ]:
# Show the per-page log so you can confirm runtimes / token counts look sane.
with open(f"{RUN_DIR}/run.log.jsonl") as f:
    for line in f:
        print(line.rstrip())

In [ ]:
# Show one page's raw model output (markdown) and one per-page JSON record.
import glob, json
for path in sorted(glob.glob(f"{RUN_DIR}/raw/*.md"))[:1]:
    print("---", path, "---")
    with open(path) as f:
        print(f.read())
for path in sorted(glob.glob(f"{RUN_DIR}/pages/*.json"))[:1]:
    print("\n---", path, "---")
    print(json.dumps(json.load(open(path)), indent=2))

## 10. Download outputs to your machine

In [ ]:
import shutil
ARCHIVE = "/content/colab_real_2b_low_v1.zip"
shutil.make_archive(ARCHIVE.replace(".zip", ""), "zip", RUN_DIR)
print("Archive:", ARCHIVE)
try:
    from google.colab import files  # type: ignore
    files.download(ARCHIVE)
except Exception as e:
    print("Auto-download skipped (not running in Colab UI):", e)

## Troubleshooting

| Symptom | Likely fix |
|---|---|
| `ModuleNotFoundError: No module named 'adaptive_inference'` | Re-run cell 3 (`pip install -e .`); make sure cell 2 changed dir into the repo. |
| `OSError: ...flash_attn...` during model load | Reload the runtime and re-run, OR add `!pip install flash-attn --no-build-isolation` (slow, ~10 min compile). InternVL2-2B should not require it; this error usually means `transformers` version drift. |
| `CUDA out of memory` | Switch to a larger GPU (L4/A100). 2B should fit on a T4 (16 GB), but if you also kept the model from a previous cell loaded, restart the runtime. |
| `RuntimeError: "..." not implemented for 'BFloat16'` | Your GPU does not support bf16. The adapter auto-detects this and uses fp16 — if you still see the error, force it: `adapter = InternVL2Adapter(model_name="internvl2-2b", model_id=cfg.model_id, dtype="float16")`. |
| Output looks garbled / very short | Expected. Fixture images are 8×8 placeholders; the model has nothing to read. Real images are the next step. |
| `git clone` fails / private repo | Edit `GIT_URL` in cell 2 to your fork, or upload the repo manually via the file browser. |